In [1]:
from openff.toolkit import Molecule, ForceField
from openff.interchange import Interchange
import openmm.app
import openmm
import openmm.unit 

try:
    del i1, i2, i3
except (UnboundLocalError, NameError):
    pass

In [2]:
inpcrd1 = openmm.app.AmberInpcrdFile('monomer.inpcrd')
prmtop1 = openmm.app.AmberPrmtopFile('monomer.prmtop', 
                                     periodicBoxVectors=inpcrd1.boxVectors)
system1 = prmtop1.createSystem(nonbondedMethod=openmm.app.PME, 
                               nonbondedCutoff=9 * openmm.unit.angstrom, 
                               switchDistance=8 * openmm.unit.angstrom,
                               constraints=openmm.app.HBonds)
i1 = Interchange.from_openmm(system1, 
                             prmtop1.topology, 
                             positions=inpcrd1.positions, 
                             box_vectors=inpcrd1.boxVectors)

In [3]:
inpcrd2 = openmm.app.AmberInpcrdFile('ALA_ALA.inpcrd')
prmtop2 = openmm.app.AmberPrmtopFile('ALA_ALA.prmtop', 
                                     periodicBoxVectors=inpcrd2.boxVectors)
system2 = prmtop2.createSystem(nonbondedMethod=openmm.app.PME, 
                               nonbondedCutoff=9 * openmm.unit.angstrom, 
                               switchDistance=8 * openmm.unit.angstrom,
                               constraints=openmm.app.HBonds)
i2 = Interchange.from_openmm(system2, 
                             prmtop2.topology, 
                             positions=inpcrd2.positions, 
                             box_vectors=inpcrd2.boxVectors)

In [4]:
offmol = Molecule.from_smiles("CCO")
offmol.generate_conformers()
ff = ForceField('openff-2.3.0.offxml')
i3 = ff.create_interchange(offmol.to_topology())

In [5]:
comb_interchange = i1.combine(i2.combine(i3))

/Users/mattthompson/software/openff-interchange/openff/interchange/operations/_combine.py:104: InterchangeCombinationWarning: Interchange object combination is complex and may produce strange results outside of use cases it has been tested in. Use with caution and thoroughly validate results!
  warnings.warn(
/Users/mattthompson/software/openff-interchange/openff/interchange/operations/_combine.py:84: InterchangeCombinationWarning: Found electrostatics 1-4 scaling factors of 5/6 with slightly different rounding (0.833333 and 0.8333333333). This likely stems from OpenFF using more digits in rounding 1/1.2. The value of 0.8333333333 will be used, which may or may not introduce small errors. 
  warnings.warn(
/Users/mattthompson/software/openff-interchange/openff/interchange/operations/_combine.py:132: UserWarning: One Interchange object has collection with name ImproperTorsions not found in the other Interchange object, but it has now been added.
  warnings.warn(


In [6]:
integrator = openmm.LangevinIntegrator(
    300 * openmm.unit.kelvin,
    1 / openmm.unit.picosecond,
    0.002 * openmm.unit.picoseconds,
)

simulation = comb_interchange.to_openmm_simulation(integrator=integrator)

In [7]:
comb_interchange = i3.combine(i2.combine(i1))

UnsupportedCombinationError: Combination failed due to key collision on id='Constraint0_DUPLICATE' mult=None associated_handler=None bond_order=None virtual_site_type=None cosmetic_attributes={}. Some keys already have _DUPLICATE appended to their ID. Please report this issue if you need this functionality.